# Explore the data

A read-only tour of the SkullFix training set: the raw volumes on disk, and the
point-cloud cache that training and evaluation actually consume.

**Nothing here writes anything.** The cache is built once by
`src/data/prepare_skullfix.py`; see the Data section of the top-level README for
how to obtain the dataset and rebuild it. This notebook only inspects the result.

Run in order — section 1 defines the paths everything below uses.

In [ ]:
import glob
import os
import sys

REPO = os.path.abspath(".")
while REPO != os.path.dirname(REPO) and not os.path.isdir(os.path.join(REPO, "src", "models")):
    REPO = os.path.dirname(REPO)
assert os.path.isdir(os.path.join(REPO, "src", "models")), f"repo root not found from {os.getcwd()}"
sys.path.insert(0, os.path.join(REPO, "src", "data"))

import paths                    # every data path is written down once
import nrrd
import numpy as np
import plotly.graph_objects as go

RAW    = os.path.join(REPO, paths.RAW_ROOT)
LABELS = os.path.join(REPO, "experiments_log", "defect_mask_labels.npz")
print("repo:", REPO)
print("raw :", RAW)

## 1 · What is on disk

Three folders, one volume per skull, paired by filename.

`implant` is the piece that was cut out. It is the ground truth for the defect
region and went unused until 2026-08-28, when the evaluation stopped inferring
that region from a distance rule and started reading it from here instead.

In [ ]:
for sub in ("complete_skull", "defective_skull", "implant"):
    f = sorted(glob.glob(os.path.join(RAW, sub, "*.nrrd")))
    print(f"{sub:16} {len(f):3} files   {os.path.basename(f[0])} ... {os.path.basename(f[-1])}")

## 2 · One volume

Binary masks on a 512x512xZ grid. Two things in the header matter downstream:

- **Voxels are not cubes and the axes are sheared** — note the off-diagonal terms
  in `space directions`. Treating indices as coordinates stretches the skull
  along one axis, so the cache pipeline multiplies by this matrix first.
- Z differs per skull, so the volumes are not all the same shape.

In [ ]:
SID = "000"
vol, hdr = nrrd.read(os.path.join(RAW, "complete_skull", f"{SID}.nrrd"))

print("shape       ", vol.shape, vol.dtype)
print("values      ", np.unique(vol))
print("bone voxels ", f"{(vol > 0).mean():.4%}")
print("\nspace directions — row i is the physical vector for one step along axis i:")
print(np.asarray(hdr["space directions"]))
print("\nvoxel size  ", np.linalg.norm(np.asarray(hdr["space directions"]), axis=0).round(3), "mm")

## 3 · Volume to surface

Marching cubes turns the binary mask into a triangle mesh. This is the first step
of the cache pipeline and the only one worth looking at by eye.

In [ ]:
import trimesh
from skimage import measure

verts, faces, normals, _ = measure.marching_cubes(vol, level=0.5)
verts_mm = verts @ np.asarray(hdr["space directions"], dtype=np.float64)   # indices -> mm
mesh = trimesh.Trimesh(vertices=verts_mm, faces=faces, vertex_normals=normals)
print(f"{len(mesh.vertices):,} vertices, {len(mesh.faces):,} faces, watertight={mesh.is_watertight}")

STEP = 4                                    # decimate so the browser stays responsive
go.Figure(go.Mesh3d(x=mesh.vertices[:, 0], y=mesh.vertices[:, 1], z=mesh.vertices[:, 2],
                    i=mesh.faces[::STEP, 0], j=mesh.faces[::STEP, 1], k=mesh.faces[::STEP, 2],
                    color="lightgray")
          ).update_layout(title=f"skull {SID} — complete, from marching cubes",
                          scene=dict(aspectmode="data"), height=520,
                          margin=dict(l=0, r=0, b=0, t=40)).show()

## 4 · The prepared cache

One `.npz`, four arrays, 12 MB. This is what training and evaluation read; the raw
volumes above are only opened by the studies that need a real mesh.

`scale_mm` is the per-skull factor converting normalised units back to
millimetres. Without it nothing can be reported in mm or compared across skulls.

In [ ]:
cache = np.load(os.path.join(REPO, paths.DATA_CACHE))
for k in cache.files:
    print(f"{k:10} {str(cache[k].shape):18} {cache[k].dtype}")

ids, inputs, gt, scale = cache["ids"], cache["inputs"], cache["gt"], cache["scale_mm"]
print(f"\nids are fixed-width strings, so leading zeros survive: {ids[:4].tolist()}")
print(f"scale_mm   mean {scale.mean():.2f}   range {scale.min():.2f}-{scale.max():.2f}")

## 5 · Input, ground truth, and the defect

`inputs` is the defective skull the model sees (4096 points); `gt` is the complete
one it must produce (6144).

Both were normalised by **one** transform, derived from the defective cloud alone
— the only shape available at inference time. Normalising them separately puts the
pair in different frames and silently destroys the correspondence; that bug was
shipped once here and fixed.

In [ ]:
j = int(np.where(ids == SID)[0][0])
mask = np.load(LABELS)[SID]
print(f"skull {SID}: {mask.sum()} of {len(mask)} ground-truth points are defect ({mask.mean():.1%})")

fig = go.Figure()
fig.add_scatter3d(x=inputs[j][:, 0], y=inputs[j][:, 1], z=inputs[j][:, 2], mode="markers",
                  marker=dict(size=1.4, color="lightsteelblue"), name="input (defective)")
fig.add_scatter3d(x=gt[j][mask][:, 0], y=gt[j][mask][:, 1], z=gt[j][mask][:, 2], mode="markers",
                  marker=dict(size=2.4, color="crimson"), name="defect region")
fig.update_layout(title=f"skull {SID} — blue is given, red is what the model must invent",
                  scene=dict(aspectmode="data"), height=560,
                  margin=dict(l=0, r=0, b=0, t=40)).show()

## 6 · Across all 100

Two numbers to know before reading any result.

The defect share explains why a whole-cloud metric is misleading here: most of the
ground truth is surface the input already shows, so a model scores well by copying.
The point spacing is the resolution the entire project is limited by — for scale,
the AutoImplant challenge works on 0.45 mm voxels.

In [ ]:
from scipy.spatial import cKDTree

labels = np.load(LABELS)
frac = np.array([labels[s].mean() for s in ids])

spacing = []
for k in range(0, len(ids), 10):                      # every tenth skull keeps this quick
    d, _ = cKDTree(gt[k]).query(gt[k], k=2)
    spacing.append(np.median(d[:, 1]) * scale[k])

print(f"defect share of GT points   {frac.mean():.2%}   range {frac.min():.1%}-{frac.max():.1%}")
print(f"median GT point spacing     {np.mean(spacing):.2f} mm   (n={len(spacing)} skulls)")